# Fig. 6C: Gene expression changes associated with Chaphamaparvovirus infection in chicken liver samples

Run GSEA on the newly discovered and diagnosed comparisons; prepare tables for `Fig6C_plot.R`.
Place both groups' DESeq2 results in the input directory before running.


In [1]:
import pandas as pd
import gseapy as gp
from gseapy import Msigdb
import os
import numpy as np

In [2]:
# Define paths
basedir = "/path/to/project"
directory_out = "Fig6C/gsea"

# Output directory
path_out = f"{basedir}/output/revise_final/{directory_out}"

# Input directory
# DESeq2 results
input_folder_path = f"{basedir}/output/revise_final/Fig6C/deseq2"

os.makedirs(path_out, exist_ok=True)
print("saving files to:", path_out)


saving files to: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/Fig6C/gsea


In [3]:
# Load the Hallmark gene sets
gene_sets = gp.get_library(
    name="MSigDB_Hallmark_2020",
    organism="Human"
)

# Load the MSigDB C2:CGP gene sets
msig = Msigdb()

c2_cgp = msig.get_gmt(
    category="c2.cgp",
    dbver="2026.1.Hs"
)

# Add the WIELAND_UP_BY_HBV_INFECTION gene set
gene_sets["WIELAND_UP_BY_HBV_INFECTION"] = \
    c2_cgp["WIELAND_UP_BY_HBV_INFECTION"]


In [4]:
def plot_prerank(df, outdir_path, metric, gene_col, gene_sets):
    """
    Run preranked GSEA and plot the top five terms.

    metric: Column used to rank genes.
    gene_col: Column containing gene symbols.
    """
    # Input columns: gene_name_hs, gene_name_gallus, baseMean, log2FoldChange, lfcSE, stat, pvalue, padj
    df2 = df.copy()
    
    # Remove rows with missing gene names or ranking values
    df2 = df2.dropna(subset=[gene_col, metric])
    df2[gene_col] = df2[gene_col].astype(str)
    
    # For duplicate genes, keep the largest absolute ranking value
    # Break ties by p-value, then gene name
    df2["_abs"] = df2[metric].astype(float).abs()
    df2["_p"] = df2["pvalue"].astype(float).fillna(1.0)

    df2 = (df2.sort_values([gene_col, "_abs", "_p", gene_col],
                           ascending=[True, False, True, True],
                           kind="mergesort").drop_duplicates(subset=[gene_col], keep="first"))

    # Rank genes, breaking ties by p-value and gene name
    df2 = df2.sort_values([metric, "_p", gene_col],
                          ascending=[False, True, True],
                          kind="mergesort")
    rnk = df2[[gene_col, metric]]
    print("\nrnk\n", rnk.head(3))

    # Run preranked GSEA
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=gene_sets,
        outdir=f"{outdir_path}_wieland_hbv_{metric}_{gene_col}",
        seed=0,
        threads=4,
        verbose=True
    )

    # Plot the top five terms
    terms = pre_res.res2d["Term"]
    pre_res.plot(
        terms=terms[:5], 
        legend_kws={'loc': (1.15, 0)},
        ofname=f"{outdir_path}_wieland_hbv_{metric}_{gene_col}_top5_enriched.png"
    )

In [5]:
configs = {
    "metric": "stat",
    "gene_col": "gene_name_hs",
    "gene_sets": gene_sets
}

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if not(fname.endswith("_hs_gallus_and_Gene.txt")):
            continue

        # Read the DESeq2 results
        path = os.path.join(dirpath, fname)
        project = pd.read_table(path)

        # Run and plot preranked GSEA
        proname = fname.split("_")[1]
        print("\nprocessing...", proname)
        plot_prerank(project, f"{path_out}/{proname}", configs["metric"], configs["gene_col"], configs["gene_sets"])

2026-11-20 20:20:29,317 [WARNING] Duplicated values found in preranked stats: 0.65% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-11-20 20:20:29,318 [INFO] Parsing data files for GSEA.............................
2026-11-20 20:20:29,327 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-11-20 20:20:29,327 [INFO] 0051 gene_sets used for further statistical testing.....
2026-11-20 20:20:29,328 [INFO] Start to run GSEA...Might take a while..................



processing... Fujino

rnk
       gene_name_hs       stat
15673       DALRD3  16.401822
22218        MAST3  16.248276
19486         CD40  16.097841


2026-11-20 20:20:33,954 [INFO] Congratulations. GSEApy runs successfully................

2026-11-20 20:20:34,183 [WARNING] Duplicated values found in preranked stats: 1.08% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-11-20 20:20:34,183 [INFO] Parsing data files for GSEA.............................
2026-11-20 20:20:34,192 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-11-20 20:20:34,193 [INFO] 0051 gene_sets used for further statistical testing.....
2026-11-20 20:20:34,193 [INFO] Start to run GSEA...Might take a while..................



processing... PRJNA577590

rnk
       gene_name_hs      stat
11736       ATRNL1  4.864696
15087         RRAD  3.511008
5100        APCDD1  3.080048


2026-11-20 20:20:38,435 [INFO] Congratulations. GSEApy runs successfully................

2026-11-20 20:20:38,678 [WARNING] Duplicated values found in preranked stats: 1.60% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-11-20 20:20:38,679 [INFO] Parsing data files for GSEA.............................
2026-11-20 20:20:38,688 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-11-20 20:20:38,688 [INFO] 0051 gene_sets used for further statistical testing.....
2026-11-20 20:20:38,688 [INFO] Start to run GSEA...Might take a while..................



processing... PRJNA622813

rnk
       gene_name_hs       stat
8499         DDX60  32.073228
19286        ZNFX1  27.803089
10787      TNFAIP2  20.985157


2026-11-20 20:20:43,162 [INFO] Congratulations. GSEApy runs successfully................

2026-11-20 20:20:43,373 [WARNING] Duplicated values found in preranked stats: 1.29% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-11-20 20:20:43,373 [INFO] Parsing data files for GSEA.............................
2026-11-20 20:20:43,382 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-11-20 20:20:43,383 [INFO] 0051 gene_sets used for further statistical testing.....
2026-11-20 20:20:43,383 [INFO] Start to run GSEA...Might take a while..................



processing... PRJNA612882

rnk
       gene_name_hs      stat
11179          RET  4.157465
12398         SPEG  4.062659
20505         ESAM  3.823232


2026-11-20 20:20:47,795 [INFO] Congratulations. GSEApy runs successfully................



In [6]:
# Select significant terms among the top and bottom ten NES values
terms = set()

for dirpath, dirnames, filenames in os.walk(path_out):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # Select the ten highest and ten lowest NES values
        top10   = res.sort_values("NES", ascending=False).head(10).copy()
        worst10 = res.sort_values("NES", ascending=True).head(10).copy()
        df = pd.concat([top10, worst10], ignore_index=True).sort_values("NES", ascending=True).reset_index(drop=True)

        # Collect terms with FDR < 0.05
        for t in df.loc[df["FDR q-val"] < 0.05, "Term"].to_list():
            terms.add(t)

# Check the number of selected terms
print("top 10 and FDR<0.05:", len(terms)) 

# Sort the selected terms
terms = sorted(list(terms))

top 10 and FDR<0.05: 31


In [7]:
# Order terms by mean NES
first_file = True

for dirpath, dirnames, filenames in os.walk(path_out):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # Keep the selected terms
        df_term = res.loc[res["Term"].isin(terms), ["Term", "NES"]].copy()
        df_term.columns = df_term.columns.map(lambda x: f"{x}_{project}" if x != "Term" else x)
        print(f"Project: {project} size: {df_term.shape}")

        # Merge results across projects
        if first_file:
            df_all = df_term
            first_file = False
        else:
            df_all = df_all.merge(df_term, on="Term", how="left")

# Check dimensions
print("df_all size:", df_all.shape)

# Sort terms by mean NES
cols = df_all.set_index("Term").columns.to_list()

df_all["mean_NES"] = df_all[cols].mean(axis=1)
df_sorted = df_all.sort_values("mean_NES", ascending=True)

# Store the term order
terms_order = df_sorted["Term"].to_list()

# Preview results
print(terms_order)

Project: PRJNA577590 size: (31, 2)
Project: Fujino size: (31, 2)
Project: PRJNA612882 size: (31, 2)
Project: PRJNA622813 size: (31, 2)
df_all size: (31, 5)
['Xenobiotic Metabolism', 'Bile Acid Metabolism', 'Pperoxisome', 'Fatty Acid Metabolism', 'Oxidative Phosphorylation', 'heme Metabolism', 'Adipogenesis', 'Myc Targets V2', 'Androgen Response', 'Protein Secretion', 'mTORC1 Signaling', 'Reactive Oxygen Species Pathway', 'Coagulation', 'Wnt-beta Catenin Signaling', 'p53 Pathway', 'UV Response Dn', 'KRAS Signaling Up', 'Hedgehog Signaling', 'Inflammatory Response', 'Epithelial Mesenchymal Transition', 'TNF-alpha Signaling via NF-kB', 'Spermatogenesis', 'Allograft Rejection', 'PI3K/AKT/mTOR  Signaling', 'IL-6/JAK/STAT3 Signaling', 'Interferon Gamma Response', 'WIELAND_UP_BY_HBV_INFECTION', 'Mitotic Spindle', 'Interferon Alpha Response', 'E2F Targets', 'G2-M Checkpoint']


In [8]:
# Create the dot plot table
dfs = []

for dirpath, dirnames, filenames in os.walk(path_out):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # Calculate −log10(FDR), clipping FDR at 1e-10
        res["mlog10_fdr"] = -np.log10(res["FDR q-val"].clip(lower=1e-10))

        # Keep the selected terms
        df_term = res.loc[res["Term"].isin(terms), ["Term", "NES", "FDR q-val", "mlog10_fdr"]].copy()
        df_term["Project"] = project
        df_term_sorted = df_term.sort_values(
            "Term",
            key=lambda s: pd.Categorical(s, categories=terms_order, ordered=True)
        )

        dfs.append(df_term_sorted)

# Combine results
df_cat = pd.concat(dfs, ignore_index=True)
# Save tables
df_cat.to_csv(f"{path_out}/NES_top10_FDR05.csv")
# Preview results
print("df_cat size:", df_cat.shape)
df_cat.head(3)

df_cat size: (124, 5)


,Term,NES,FDR q-val,mlog10_fdr,Project
0,Xenobiotic Metabolism,-1.033544,0.683936,0.164984,PRJNA577590
1,Bile Acid Metabolism,-1.016422,0.693901,0.158702,PRJNA577590
2,Pperoxisome,-0.972732,0.741377,0.129961,PRJNA577590
